# Neural Network Model

We will be training three types of neural network models
1. Temporal only Neural network model (Baseline)
2. Temporal + Static Neural Network (Single Stage Model)
3. Temporal Neural Network on Trend, Static Random Forest on Residuals (Two Stage Models)

This notebook illustrates the training process for the baseline neural network model on only the temporal features (Type 1). This model is used simultaneously as a baseline for the neural network models, and also as an input to the two stage models.

The network sees only the temporal features of the month a flat was approved in (the 177 stock, COE and macroeconomic features of `data/feature_config_temporal.py`, from `data/features/final/temporal_scaled.csv` (already scaled by the data pipeline, so the network does no scaling of its own)), so every flat of a month gets the same prediction: the level of rent in that month.

Some details on the model architecture:

* **Network:**  We use an encoder like structure, with reducing hidden layer sizes to do automatic feature selection. The reducing layer size acts like a dimensionality reducer, much like PCA, with the encoding mechanism learned from gradient descent instead of computed via eigenvector decomposition.

* **Architecture** nn.ModuleList(`177 -> 64 -> 16 -> 1`), LeakyReLU activation between layers.

* **Output** The raw monthly rent as the target (standardised inside the training loop). Final layer has no activation.

* **Training Parameters** Adam Optimiser with learning rate 1e-3, batch size 2048, 100 epochs, random seed 0.

* **Sample Use:** Train on the `train` months of the final feature tables (2021-01..2024-03), evaluate on `val` months (2024-04..2025-03). Run a final fit on train + validation (2021-01..2025-03), followed by inference on all months. Returns monthly trend levels.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1]))     # 02_modelling: nn_utils.py is shared by the single-stage and two-stage notebooks
import nn_utils as nu

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

NAME = 'temporal_ffn'
HIDDEN = [64, 16]
ACTIVATION = 'leaky_relu'            # 'leaky_relu' or 'tanh'
EPOCHS, BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, SEED = 100, 2048, 1e-3, 0.0, 0
nu.use_results_dir(Path.cwd() / 'results')
nu.ensure_output_dirs(NAME)
fit_args = dict(hidden=HIDDEN, activation=ACTIVATION, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, seed=SEED)

## 1. Data

The temporal features of the 67 months (2021-01..2026-07), and the month and rent of the train + validation rows. The test rows only need their month.

In [2]:
temporal = nu.load_temporal()
FEATURES = list(temporal.columns)
train, test, _ = nu.load_static()                                  # the static features are not inputs of this network; the frames carry ROW_ID, the month and the identifier columns
fit, val = (train['sample_type'] == 'train').to_numpy(), (train['sample_type'] == 'val').to_numpy()
month, y = train['month'].to_numpy(), train[nu.LABEL].to_numpy()

print(f'{len(FEATURES)} temporal features; train rows {fit.sum():,} (months 0..{nu.TRAIN_LAST}), validation rows {val.sum():,} (months {nu.TRAIN_LAST + 1}..{nu.VALIDATION_LAST}), '
      f'test rows {len(test):,} (months {test["month"].min()}..{test["month"].max()})')

177 temporal features; train rows 114,841 (months 0..38), validation rows 35,158 (months 39..50), test rows 50,000 (months 50..66)


## 2. Held-out fit: 2021-01..2024-03

Trained on the train rows only. The validation rows are never trained on; their RMSE is logged after every epoch (`epoch_losses_heldout.csv`).

In [3]:
held = nu.fit_ffn(temporal, month[fit], y[fit], valid=(month[val], None, y[val]), **fit_args)
pred_val = nu.predict_ffn(held, temporal, month[val])
print(held['history'].iloc[[0, 9, 19, 49, 99]].round(1).to_string(index=False))

 epoch  train_mse_std  train_rmse_dollar  valid_rmse_dollar
     1            0.7              623.8              665.3
    10            0.7              605.0              664.5
    20            0.7              605.0              670.6
    50            0.7              604.9              662.9
   100            0.7              604.6              664.4


## 3. Final fit: 2021-01..2025-03

The same network trained on all the train + validation rows, then asked for the level of every month 0..66.
Each flat of a month gets that month's level, so the level of the same flats in every month is the level itself.

In [4]:
final = nu.fit_ffn(temporal, month, y, **fit_args)
level = nu.month_levels(final, temporal)                              # months 0..66
pred_train, pred_test = level.loc[month].to_numpy(), level.loc[test['month']].to_numpy()
print(final['history'].iloc[[0, 9, 19, 49, 99]].round(1).to_string(index=False))
print(level.loc[44:].round(0).rename(lambda m: nu.month_label(m)[0]).to_frame('level').T.to_string())

 epoch  train_mse_std  train_rmse_dollar
     1            0.7              634.1
    10            0.7              618.1
    20            0.7              617.9
    50            0.7              617.8
   100            0.7              617.8
       2024-09  2024-10  2024-11  2024-12  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07
level   3088.0   3093.0   3117.0   3099.0   3103.0   3113.0   3138.0   3102.0   3180.0   3173.0   3214.0   3177.0   3165.0   3205.0   3175.0   3175.0   3183.0   3205.0   3184.0   3251.0   3274.0   3321.0   3325.0


## 4. Save the model, the training statistics and the predictions

In [5]:
paths = [nu.MODEL_DIR / f'{NAME}.pt']
nu.save_ffn(final, paths[0], temporal_features=FEATURES)
paths += [nu.save_table(held['history'], NAME, 'epoch_losses_heldout.csv', index=False), nu.save_table(final['history'], NAME, 'epoch_losses_final.csv', index=False)]
paths += [nu.save_predictions(NAME, 'val', train[val], pred_val),       # held-out: the network fitted on 2021-01..2024-03
          nu.save_predictions(NAME, 'train', train, pred_train),        # in-sample: the final network, all 149,999 train + validation rows
          nu.save_predictions(NAME, 'test', test, pred_test),           # the final network, the 50,000 test rows
          nu.save_levels(NAME, level)]
print('saved:', *paths, sep='\n  ')

saved:
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/neural_network/results/models/temporal_ffn.pt
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/neural_network/results/tables/temporal_ffn/epoch_losses_heldout.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/neural_network/results/tables/temporal_ffn/epoch_losses_final.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/val/temporal_ffn.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/train/temporal_ffn.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/test/temporal_ffn.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/levels/temporal_ffn.csv
